# 13 — Prototype: bar-frequency volatility HMM

**Development-only prototype.** This notebook implements the first Module 13 switching test after discussions/010–011.

The main design changes relative to the pre-Module-13 sketch are:

1. **30m/60m overlapping targets are not HMM emissions.** The HMM observes one-bar index variance, \(y_t=\log(r_t^2+\epsilon)\), so overlap cannot manufacture state persistence.
2. **Student-t emissions from the start.** A Gaussian scale-switching HMM can create a fake high-volatility state merely to absorb fat tails.
3. **Two non-switching nulls.** Switching is compared with both a static K=1 bar-HAR-t and an expanding, session-by-session adaptive K=1 bar-HAR-t.
4. **K=2 only.** K-selection and HSMM durations are not tested until K=2 earns OOS value.
5. **Same Module 12 economic targets.** The bar model recursively produces expected 30m/60m integrated variance and is scored with QLIKE on the strict Module 12 index-RV targets.

Models:

- **K1**: all mean coefficients shared; one state.
- **S1**: K=2, state-specific scale only.
- **S2**: K=2, state-specific intercept + scale.
- **S3**: K=2, state-specific HAR persistence + intercept + scale; clock/weekend shared.

No state is given an economic label in this notebook. Any candidate switching gain must later beat a HAR-GARCH-t no-regime simulation null with at least 200 paths before state interpretation.


In [ ]:
from pathlib import Path
import os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 1000)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling import vol_hmm as vhm
from data_engineering.modeling.hsmm.session_eval import paired_sessions

ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
DEV_START, SCREEN_TRAIN_END, DEV_END = "2026-09-22", "2026-09-28", "2026-10-07"
HORIZONS = list(vt.HORIZONS)
MODELS = ["STATIC_K1", "ADAPT_K1", "S1", "S2", "S3"]
N_BOOT = int(os.getenv("M13_BOOT", "2000"))
N_INIT = int(os.getenv("M13_N_INIT", "4"))
MAX_ITER = int(os.getenv("M13_MAX_ITER", "150"))
SEED = int(os.getenv("M13_SEED", "1300"))

print(f"vol_hmm {vhm.VOL_HMM_VERSION} | n_init {N_INIT} | max_iter {MAX_ITER} | boot {N_BOOT}")


## 1. Development data only

Module 13 specification work is restricted to sessions through 2026-10-07. Module 12's frozen confirmation specification is untouched.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
required = {"state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend", "in_settlement_window", "option_atm_quote_ok", "option_iv_atm_forward"}
missing = sorted(required - cols)
if missing:
    raise RuntimeError(f"time_series_feature_state lacks {missing}")

DEV_LAST_BAR = pd.Timestamp(f"{DEV_END} 08:00", tz="UTC")
want = sorted(required)
df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state "
    f"WHERE state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}' "
    "ORDER BY underlying, state_time"
).df()
con.close()

df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["session_date"] = ri.session_key(df["state_time"])
keep = ((df["session_date"] >= DEV_START) & (df["session_date"] <= DEV_END)) | (df["state_time"] == DEV_LAST_BAR)
df = df[keep].reset_index(drop=True)

DEV_SESSIONS = sorted(s for s in df["session_date"].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}

train_weekend = df["session_date"].isin(SCREEN_TRAIN) & df["is_weekend"].fillna(False).astype(bool)
if not train_weekend.any():
    raise RuntimeError("Initial Module 13 training set must contain weekend observations.")

print(f"rows {len(df):,} | development sessions {len(DEV_SESSIONS)} | train {len(SCREEN_TRAIN)} | eval {len(SCREEN_EVAL)}")
print("train:", sorted(SCREEN_TRAIN))
print("eval :", sorted(SCREEN_EVAL))


## 2. Fit static K1 and K=2 S1/S2/S3; build expanding K1 benchmark

The K=2 models are deliberately trained only on the initial 2026-09-22 → 09-28 window. The adaptive K1 benchmark is re-estimated before each evaluation session using every completed earlier development session. This makes the switching gate conservative: a static HMM must beat a non-switching model that is allowed to adapt to level drift.


In [ ]:
def asset_frame(asset):
    return df[df["underlying"] == asset].sort_values("state_time").reset_index(drop=True)


STATIC_FITS = {}
TRAIN_DATA = {}
SCORE_FRAMES = []
STATE_ROWS = []
EPISODE_ROWS = []

t_all = time.perf_counter()
for ai, asset in enumerate(ASSETS):
    g = asset_frame(asset)
    panel = vhm.build_panel(g, SCREEN_TRAIN)
    score_eps = dict(panel.score_eps)
    STATIC_FITS[asset] = {}
    TRAIN_DATA[asset] = {}

    # Static one-state benchmark and static K=2 switching prototypes.
    for mi, structure in enumerate(["K1", "S1", "S2", "S3"]):
        fit, tr_data = vhm.fit_model(
            panel, SCREEN_TRAIN, structure,
            K=1 if structure == "K1" else 2,
            seed=SEED + 100 * ai + mi,
            n_init=N_INIT,
            max_iter=MAX_ITER,
        )
        STATIC_FITS[asset][structure] = fit
        TRAIN_DATA[asset][structure] = tr_data
        name = "STATIC_K1" if structure == "K1" else structure

        pt = vhm.parameter_table(fit)
        pt.insert(0, "underlying", asset)
        pt.insert(1, "model", name)
        STATE_ROWS.append(pt)

        ep = vhm.episode_table(fit, tr_data)
        if len(ep):
            es = ep.groupby("state").agg(
                episodes=("length", "size"),
                median_length=("length", "median"),
                mean_length=("length", "mean"),
            ).reset_index()
            es.insert(0, "underlying", asset)
            es.insert(1, "model", name)
            EPISODE_ROWS.append(es)

        ev_data, filt = vhm.filter_model(fit, panel, SCREEN_EVAL)
        for h, m in vt.HORIZONS.items():
            f = vhm.forecast_integrated_variance(fit, panel, ev_data, filt, m)
            sc = vhm.score_forecasts(panel, ev_data, f, h, name)
            sc.insert(0, "underlying", asset)
            SCORE_FRAMES.append(sc)

    # Expanding one-state benchmark. Refit only before each new session.
    for si, session in enumerate(sorted(SCREEN_EVAL)):
        prior = {s for s in DEV_SESSIONS if s < session}
        p_ad = vhm.build_panel(g, prior, score_eps=score_eps)
        fit_ad, _ = vhm.fit_model(
            p_ad, prior, "K1", K=1,
            seed=SEED + 10000 + 100 * ai + si,
            n_init=1,
            max_iter=MAX_ITER,
        )
        ev_ad, filt_ad = vhm.filter_model(fit_ad, p_ad, {session})
        for h, m in vt.HORIZONS.items():
            f = vhm.forecast_integrated_variance(fit_ad, p_ad, ev_ad, filt_ad, m)
            sc = vhm.score_forecasts(p_ad, ev_ad, f, h, "ADAPT_K1")
            sc.insert(0, "underlying", asset)
            SCORE_FRAMES.append(sc)

    print(f"{asset}: complete in {time.perf_counter() - t_all:.0f}s cumulative")

scores = pd.concat(SCORE_FRAMES, ignore_index=True)
state_params = pd.concat(STATE_ROWS, ignore_index=True)
episodes = pd.concat(EPISODE_ROWS, ignore_index=True) if EPISODE_ROWS else pd.DataFrame()
print(f"score rows: {len(scores):,}")

## 3. Descriptive forecast metrics

These are development-screening metrics. The switching gate is the paired comparison in the next section, not the standalone R²/QLIKE table.


In [ ]:
metrics = (
    scores.dropna(subset=["qlike"])
    .groupby(["underlying", "horizon", "model"])
    .agg(
        n=("qlike", "size"),
        qlike=("qlike", "mean"),
        log_rmse=("sqerr", lambda x: float(np.sqrt(np.mean(x)))),
    )
    .reset_index()
)
display(metrics.round(6))

## 4. Session-paired switching gate

Positive QLIKE difference means the first model is better. A candidate switching result must beat **both** STATIC_K1 and ADAPT_K1. The non-overlapping score view must agree in sign; a bootstrap interval above zero is reported but remains development evidence only.


In [ ]:
def compare_pair(asset, horizon, a_model, b_model, view):
    A = scores[(scores.underlying == asset) & (scores.horizon == horizon) & (scores.model == a_model)].copy()
    B = scores[(scores.underlying == asset) & (scores.horizon == horizon) & (scores.model == b_model)].copy()
    d = A.merge(B, on=["underlying", "raw_index", "session", "horizon"], suffixes=("_a", "_b"))
    ok = np.isfinite(d["qlike_a"]) & np.isfinite(d["qlike_b"])
    if view == "non_overlapping":
        step = vt.HORIZONS[horizon] * vt.BAR_MIN
        ok &= np.isclose(np.mod(np.round(d["tte_min_a"].to_numpy()), step), 0.0)
    d = d.loc[ok].copy()
    if d.empty:
        return {"n_rows": 0, "n_sessions": 0, "mean_session_diff": np.nan, "boot_lo": np.nan, "boot_hi": np.nan, "sessions_won": np.nan}
    diff = d["qlike_b"].to_numpy() - d["qlike_a"].to_numpy()
    p = paired_sessions(diff, np.zeros_like(diff), d["session"].to_numpy(), n_boot=N_BOOT)
    return {
        "n_rows": len(d),
        "n_sessions": p["n_sessions"],
        "mean_session_diff": p["mean_session_diff"],
        "boot_lo": p["boot_lo"],
        "boot_hi": p["boot_hi"],
        "sessions_won": p["sessions_won"],
    }

pairs = []
for asset in ASSETS:
    for h in HORIZONS:
        for model in ["S1", "S2", "S3"]:
            for benchmark in ["STATIC_K1", "ADAPT_K1"]:
                for view in ["all", "non_overlapping"]:
                    pairs.append({"underlying": asset, "horizon": h, "model": model, "vs": benchmark, "view": view, **compare_pair(asset, h, model, benchmark, view)})
        for model, benchmark in [("S2", "S1"), ("S3", "S2")]:
            for view in ["all", "non_overlapping"]:
                pairs.append({"underlying": asset, "horizon": h, "model": model, "vs": benchmark, "view": view, **compare_pair(asset, h, model, benchmark, view)})

comparisons = pd.DataFrame(pairs)
display(comparisons.round(6))

OUT = PROJECT_ROOT / "notebooks"
metrics.to_csv(OUT / "13_screening_metrics.csv", index=False)
comparisons.to_csv(OUT / "13_screening_comparisons.csv", index=False)
state_params.to_csv(OUT / "13_state_parameters.csv", index=False)
episodes.to_csv(OUT / "13_state_episodes.csv", index=False)
print("wrote Module 13 prototype CSV outputs")


## 5. Anonymous state diagnostics

These diagnostics describe the fitted training states only. Do not call them “low vol”, “shock”, MR, MOM, or Bridge yet. The first legitimate labels come only after the switching forecast gate and the no-regime null pass.


In [ ]:
display(state_params.round(5))
if len(episodes):
    display(episodes.round(3))


## 6. Prototype gate and next required test

A model/horizon is only a **candidate for the no-regime null** if its mean session QLIKE improvement is positive against both K1 benchmarks in both score views. This deliberately weak development gate does not declare a regime.

If any candidate appears, the next implementation step is the mandatory HAR-GARCH-t simulation null from discussion/011 §5.8, with at least 200 paths and the real missingness/session/clock pattern. If no candidate appears, the switching branch stops before HSMM.


In [ ]:
gate_rows = []
for asset in ASSETS:
    for h in HORIZONS:
        for model in ["S1", "S2", "S3"]:
            d = comparisons[(comparisons.underlying == asset) & (comparisons.horizon == h) & (comparisons.model == model) & (comparisons["vs"].isin(["STATIC_K1", "ADAPT_K1"]))]
            tests = {(r["vs"], r["view"]): r for _, r in d.iterrows()}
            required = [(b, v) for b in ["STATIC_K1", "ADAPT_K1"] for v in ["all", "non_overlapping"]]
            pass_sign = all(k in tests and np.isfinite(tests[k]["mean_session_diff"]) and tests[k]["mean_session_diff"] > 0 for k in required)
            pass_ci = all(k in tests and np.isfinite(tests[k]["boot_lo"]) and tests[k]["boot_lo"] > 0 for k in required)
            gate_rows.append({"underlying": asset, "horizon": h, "model": model, "positive_all_four": pass_sign, "ci_above_zero_all_four": pass_ci})

gates = pd.DataFrame(gate_rows)
display(gates)
cand = gates[gates["positive_all_four"]]
if len(cand):
    print("Candidate switching cases require HAR-GARCH-t null >= 200 paths before interpretation:")
    display(cand)
else:
    print("No switching structure beats both static and adaptive K1 in both score views. Do not proceed to HSMM.")


## 7. Prototype limitations

This first implementation intentionally stops before three expensive extensions:

1. **HAR-GARCH-t no-regime null** — mandatory only for candidates that survive §6.
2. **Adaptive refitting of K=2 HMMs** — the prototype gives switching a harder test by keeping K=2 static while allowing K1 to adapt. If a static K=2 candidate survives, a like-for-like expanding HMM can be added before the null.
3. **HSMM duration** — not tested until a K=2 HMM survives the two non-switching benchmarks and the no-regime null.

Module 12's `12_frozen_spec_v02.json` remains untouched throughout.
